# Module 10 - Programming Assignment

## Directions

1. Change the name of this file to be your JHED id as in `jsmith299.ipynb`. Because sure you use your JHED ID (it's made out of your name and not your student id which is just letters and numbers).
2. Make sure the notebook you submit is cleanly and fully executed. I do not grade unexecuted notebooks.
3. Submit your notebook back in Blackboard where you downloaded this file.

*Provide the output **exactly** as requested*

# Forward Planner

## Unify

Use the accompanying `unification.py` file for unification. For this assignment, you're almost certainly going to want to be able to:

1. specify the problem in terms of S-expressions.
2. parse them.
3. work with the parsed versions.

`parse` and `unification` work exactly like the programming assignment for last time.

In [1]:
from unification import parse, unification, is_variable
from typing import List, Dict, FrozenSet, Set, Tuple
from copy import deepcopy

## Forward Planner

In this assigment, you're going to implement a Forward Planner. What does that mean? If you look in your book, you will not find pseudocode for a forward planner. It just says "use state space search" but this is less than helpful and it's a bit more complicated than that. **(but please please do not try to implement STRIPS or GraphPlan...that is wrong).**

At a high level, a forward planner takes the current state of the world $S_0$ and attempts to derive a plan, basically by Depth First Search. We have all the ingredients we said we would need in Module 1: states, actions, a transition function and a goal test. We have a set of predicates that describe a state (and therefore all possible states), we have actions and we have, at least, an implicit transition function: applying an action in a state causes the state to change as described by the add and delete lists.

Let's say we have a drill that's an item, two places such as home and store, and we know that I'm at home and the drill is at the store and I want to go buy a drill (have it be at home). We might represent that as:

<code>
start_state = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Saw Store)",
    "(at Drill Store)",
    "(at Money Bank)"
]
</code>

And we have a goal state:

<code>
goal = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Drill Me)",
    "(at Saw Store)",
    "(at Money Bank)"
]
</code>

The actions/operators are:

<code>
actions = {
    "drive": {
        "action": "(drive ?agent ?from ?to)",
        "conditions": [
            "(agent ?agent)",
            "(place ?from)",
            "(place ?to)",
            "(at ?agent ?from)"
        ],
        "add": [
            "(at ?agent ?to)"
        ],
        "delete": [
            "(at ?agent ?from)"
        ]
    },
    "buy": {
        "action": "(buy ?purchaser ?seller ?item)",
        "conditions": [
            "(item ?item)",
            "(place ?seller)",
            "(agent ?purchaser)",
            "(at ?item ?seller)",
            "(at ?purchaser ?seller)"
        ],
        "add": [
            "(at ?item ?purchaser)"
        ],
        "delete": [
            "(at ?item ?seller)"
        ]
    }
}
</code>

These will all need to be parsed from s-expressions to the underlying Python representation before you can use them. You might as well do it at the start of your algorithm, once. The order of the conditions is *not* arbitrary. It is much, much better for the unification and backtracking if you have the "type" predicates (item, place, agent) before the more complex ones. Trust me on this.

As for the algorithm itself, there is going to be an *outer* level of search and an *inner* level of search.

The *outer* level of search that is exactly what I describe here: you have a state, you generate successor states by applying actions to the current state, you examine those successor states as we did at the first week of the semester and if one is the goal you stop, if you see a repeat state, you put it on the explored list (you should implement graph search not tree search). What could be simpler?

It turns out the Devil is in the details. There is an *inner* level of search hidden in "you generate successor states by applying actions to the current state". Where?

How do you know if an action applies in a state? Only if the preconditions successfully unify with the current state. That seems easy enough...you check each predicate in the conditions to see if it unifies with the current state and if it does, you use the substitution list on the action, the add and delete lists and create the successor state based on them.

Except for one small problem...there may be more than one way to unify an action with the current state. You must essentially search for all successful unifications of the candidate action and the current state. This is where my question through the semester appliesm, "how would you modify state space search to return all the paths to the goal?"

Unification can be seen as state space search by trying to unify the first precondition with the current state, progressively working your way through the precondition list. If you fail at any point, you may need to backtrack because there might have been another unification of that predicate that would succeed. Similarly, as already mentioned, there may be more than one.

So...by using unification and a properly defined <code>successors</code> function, you should be able to apply graph based search to the problem and return a "path" through the states from the initial state to the goal. You'll definitely want to use graph-based search since <code>( drive Me Store), (drive Me Home), (drive Me Store), (drive Me Home), (drive Me Store), (buy Me Store Drill), (drive Me Home)</code> is a valid plan.

Your function should return the plan...a list of actions, fully instantiated, for the agent to do in order: [a1, a2, a3]. If you pass an extra intermediate=True parameter, it should also return the resulting state of each action: [s0, a1, s1, a2, s2, a3, s3].

-----

(you can just overwrite that one and add as many others as you need). Remember to follow the **Guidelines**.


-----

So you need to implement `forward_planner` as described above. `start_state`, `goal` and `actions` should all have the layout above and be s-expressions.

Your implementation should return the plan as a **List of instantiated actions**. If `debug=True`, you should print out the intermediate states of the plan as well.

<a id="deatomize"></a>
## deatomize

*`deatomize` Recursively transform an expression, represented as list of strings or sub-expressions, into a string with sub expressions parenthesized.* **Used by**: [assign](#assign)

* **expression** List - list representation of an S-expression to convert into a string representation.

**returns** str | None - string representation of the expression or None if expression could not be converted to a string

In [2]:
def deatomize(expression: List) -> str | None:
    if isinstance(expression, str):  # base-case
        return expression

    if isinstance(expression, list):  # recurse
        str_expression = []
        for subexpressions in expression:
            result = deatomize(subexpressions)
            if result is None:
                return None
            str_expression.append(result)
        return "(" + " ".join(str_expression) + ")"

    return None

In [3]:
s_expression1 = ["son", "?y", "Barney"]
assert deatomize(s_expression1) == "(son ?y Barney)"  # test 1 - no nested expressions

s_expression2 = ["son", "?y", ["son", "Barney"]]
assert deatomize(s_expression2) == "(son ?y (son Barney))"  # test 2 - single nested expression

s_expression3 = ["son", "?y", ["son", "Barney", ["love", "Fred"]]]
assert deatomize(s_expression3) == "(son ?y (son Barney (love Fred)))"  # test 3 - more than one nested expression

<a id="hash_state"></a>
## hash_state

*`hash_state` Compute a hashable state representing using [deatomize](#deatomize) to collapse the state into a flat list and then casting to a FrozenSet.* **Uses** [deatomize](#deatomize)

* **state** List - list representation of an S-expression to convert into a flat frozenset.

**returns** FrozenSet | None - frozenset of state or None if expression could not be converted to a flat string

In [4]:
def hash_state(state: List) -> FrozenSet | None:
    flat_state = [deatomize(s) for s in state]
    if None in flat_state:
        return None
    return frozenset(flat_state) # adapted from pa6

In [5]:
s_expression1 = ["son", "?y", "Barney"]
assert hash_state(s_expression1) == frozenset({"son", "?y", "Barney"}) # test 1 - no nested expressions

s_expression2 = ["son", "?y", ["son", "Barney"]]
assert hash_state(s_expression2) == frozenset({"son", "?y", "(son Barney)"}) # test 2 - single nested expression

s_expression3 = ["son", "?y", ["son", "Barney", ["love", "Fred"]]]
assert hash_state(s_expression3) == frozenset({"son", "?y", "(son Barney (love Fred))"}) # test 3 - more than one nested expression

<a id="is_goal_state"></a>
## is_goal_state

*`is_goal_state` Check if a given state matches the goal state. That is, each predicate in the goal state is also in the current state. The states are hashed before comparing, if either state cannot be hashed the function returns False.* **Uses** [hash_state](#hash_state)

* **state** List - list representation of an S-expression to compare with.
* **goal** List - list representation of an S-expression to compare with.

**returns** bool - True if state and goal match else False

In [6]:
def is_goal_state(state: List, goal: List) -> bool:
    goal_hash = hash_state(goal)
    state_hash = hash_state(state)
    if goal_hash is None or state_hash is None:
        return False
    return goal_hash.issubset(state_hash)

In [7]:
state = [["plane", "1973"], ["airport", "SFO"], ["airport", "JFK"], ["at", "1973", "SFO"]]
goal_state = [["plane", "1973"], ["airport", "SFO"], ["airport", "JFK"], ["at", "1973", "SFO"]]

assert is_goal_state(state, goal_state)  # test 1 - goal and state are the same

state = [["plane", "1973"], ["airport", "SFO"], ["airport", "JFK"], ["at", "1973", "SFO"]]
goal_state = [["plane", "1973"], ["airport", "SFO"], ["airport", "JFK"]]

assert is_goal_state(state, goal_state)  # test 1 - goal is a subset of state returns true

state = [["plane", "1973"], ["airport", "SFO"], ["at", "1973", "SFO"]]
goal_state = [["plane", "1973"], ["airport", "SFO"], ["airport", "JFK"], ["at", "1973", "SFO"]]

assert is_goal_state(state, goal_state) is False  # test 1 - state is missing predicates in goal so should return false

<a id="apply"></a>
## apply

*`apply` Using a substitution dictionary that maps variables to values, swap any occurrence of the variable in expression with its value in the map.* **Used by**: [unification](#unification)

* **substitutions** Dict - a mapping of variables to values to swap with 
* **expression** List - S-expression to replace variables in. An expression is represented as a list of strings or sub-expressions

**returns** List - modified expression 

In [8]:
def apply(substitutions: Dict, expression: List) -> List:
    return [substitutions[i] if is_variable(i) and i in substitutions else i for i in expression]

In [9]:
assert apply({"?x": "Barney"}, ["son", "?x", "Bam_Bam"]) == ["son", "Barney", "Bam_Bam"]  # test 1 - properly applies substitution
assert apply({"?y": "Fred"}, ["son", "?x", "Bam_Bam"]) == ["son", "?x", "Bam_Bam"]  # test 2 - no modification if variable not in substitution list
assert apply({"?x": "Barney"}, ["son", "?x", ["loves", "?x", "Bam_Bam"]]) == [ "son", "Barney", ["loves", "?x", "Bam_Bam"]]  # test 3 - only applies substitution in the top level of the expression

<a id="apply_action_to_state"></a>
## apply_action_to_state

*`apply_action_to_state` modify state by applying substitutions to each element in add_list and del_list and then adding to and removing from the state.* **Used by**: [unification](#unification)

* **state** List - list representation of an S-expression to compare with.
* **substitutions** Dict - a mapping of variables to values to swap with 
* **add_list** List - list of S-expression to add to state after performing substitutions
* **del_list** List - list of S-expression to remove from state after performing substitutions

**returns** List - modified state

In [10]:
def apply_action_to_state(state: List, substitutions: Dict, add_list: List, del_list: List) -> List:
    new_state = deepcopy(state)
    
    for add_item in add_list: # add items to new state
        add_item = apply(substitutions, add_item)
        if add_item not in new_state:
            new_state.append(add_item)

    for del_item in del_list: # remove items from state
        del_item = apply(substitutions, del_item)
        if del_item in new_state:
            new_state.remove(del_item)
    return new_state

In [11]:
state = []
sub = {"?x": "Barney"}
add_list = [["son", "?x", "Bam_Bam"]]
del_list = []
assert apply_action_to_state(state, sub, add_list, del_list) == [["son", "Barney", "Bam_Bam"]]  # test 1 - substitute and add from add list


state = [["son", "Barney", "Bam_Bam"]]
sub = {"?x": "Barney"}
add_list = []
del_list = [["son", "?x", "Bam_Bam"]]
assert apply_action_to_state(state, sub, add_list, del_list) == []  # test 2 - substitute and del from delete list

state = [["son", "Barney", "Bam_Bam"]]
sub = {"?x": "Barney"}
add_list = [["son", "?x", "Bam_Bam"]]
del_list = []
assert apply_action_to_state(state, sub, add_list, del_list) == [["son", "Barney", "Bam_Bam"]]  # test 3 - dont add duplicates

<a id="get_all_unifications"></a>
## get_all_unifications

*`get_all_unifications` get all possible unifications of a given state over a set of conditions using BFS. at each stage of the search, the first condition is unified against each predicate in the state, and if it unifies, the rest of the conditions and the substitution map are added to the search stack. A base-case of the search is reached if the conditions have been fully unified (no conditions remain). In this case the built-up substitution map is added to the result list. After the conditions have been matches against all possible predicates the list of fully matched substitution maps is returned.* **Used by**: [unification](#unification)

* **state** List - list representation of an S-expression to unify with
* **conditions** List - list of S-expression conditions to unify against


**returns** List[Dict] - List of substitution maps for all possible unifications of the state with the conditions

In [12]:
def get_all_unifications(state: List, conditions: List) -> List[Dict]:
    all_substitutions = []
    stack = [(conditions, {})]
    
    while stack:
        preconditions, substitutions = stack.pop(0)
        if not preconditions:  # fully matched state and conditions
            all_substitutions.append(substitutions)
            continue
        
        condition, remainder = preconditions[0], preconditions[1:] # same idea from unification alg.
        for predicate in state:
            child_substitutions = unification(predicate, condition, deepcopy(substitutions))
            if child_substitutions is not False:
                stack.append((remainder, child_substitutions))
    return all_substitutions

In [15]:
state = [["plane", "1973"], ["airport", "SFO"], ["airport", "JFK"], ["at", "1973", "SFO"]]
conditions = [
            ["plane", "?plane"],
            ["airport", "?to"],
            ["airport", "?from"],
            ["at", "?plane", "?from"]
        ]
get_all_unifications(state, conditions) 
# assert get_all_unifications(state, conditions) == [{"?plane": "1973", "?to": "SFO", "?from": "SFO"}]


[{'?plane': '1973', '?to': 'SFO', '?from': 'SFO'},
 {'?plane': '1973', '?to': 'JFK', '?from': 'SFO'}]

<a id="get_all_neighbors"></a>
## get_all_neighbors

*`get_all_neighbors` given a state and possible actions, generate all successor states from all possible unification of each actions preconditions and the state. A new successor state is made by applying the substitution map generated by the unification to the action, the add list and the delete list of the action, and adding each item from the add list to the state, and removing items from the del list from the state. Each successor state is represented as a tuple of the new state and the action used to transition from the old state to the new state* **Uses**: [get_all_unifications](#get_all_unifications), [apply](#apply) and [apply_action_to_state](#apply_action_to_state) **Used By** [planner](#forward-planner)

* **state** List - list representation of S-expressions that make up the current state 
* **actions** Dict - dictionary containing possible actions, their preconditions, add_list and del_list


**returns** List[Tuple[List, List]] - List of tuple pairs with the first element being the new state and the second element holding the action 

In [ ]:
def get_all_neighbors(state: List, actions: Dict) ->List[Tuple[List, List]]:
    successors = []
    for action in actions.values():
        for substitutions in get_all_unifications(state, action["conditions"]):
            planned_action = apply(substitutions, action["action"])
            new_state = apply_action_to_state(state, substitutions, action["add"], action["delete"])
            successors.append((new_state, planned_action))
    return successors

In [ ]:
start_state = [
    "(plane 1973)",
    "(airport SFO)",
    "(airport JFK)",
    "(at 1973 SFO)"
]


actions = {
    "fly": {
        "action": "(fly ?plane ?from ?to)",
        "conditions": [
            "(plane ?plane)",
            "(airport ?to)",
            "(airport ?from)",
            "(at ?plane ?from)"
        ],
        "add": [
            "(at ?plane ?to)"
        ],
        "delete": [
            "(at ?plane ?from)"
        ]
    }
}

# pstart_state, _, pactions = parse_inputs(start_state,start_state,actions )
# result = get_all_neighbors(pstart_state, pactions)
# for r in result: 
#     print(r)


start_state = [
    "(plane 1973)",
    "(plane 2749)",
    "(airport SFO)",
    "(airport JFK)",
    "(airport ORD)",
    "(at 1973 SFO)",
    "(at 2749 JFK)",
    "(at 97 ORD)",
    "(at 1211 SFO)"
]
# pstart_state, _, pactions = parse_inputs(start_state,start_state, actions)
# result = get_all_neighbors(pstart_state, pactions)
# for s,a in result: 
#     print(s,a)

<a id="get_plan"></a>
## get_plan

*`get_plan` given an end state, a plan of actions and a list of states needed to get to the end state, return a plan. If debug is False, a list of actions, each represented as a string, is returned. With debug enabled, the states resulting from applying each action is also returned.* **Uses** [deatomize](#deatomize) **Used by** [forward_planner](#forward-planner)

* **current_state** List - list representation of S-expressions that make up the current state 
* **actions** Dict - dictionary containing possible actions, their preconditions, add_list and del_list
* **plan** List - list of actions taken by the search
* **path** List - list of states searched by the planner
* **debug** bool - if False return list of actions, if true return states as well 


**returns** List - List of actions, and if debug is enabled, list of actions and states

In [ ]:
def get_plan(current_state: List, plan: List, path: List, debug: bool = False) -> List:
    if not debug:            
        return [deatomize(i) for i in plan]
    full_plan = []
    for action, state in zip(plan, path):
        full_plan.append([deatomize(pred) for pred in state])
        full_plan.append(deatomize(action))
    full_plan.append([deatomize(pred) for pred in current_state])
    return full_plan

<a id="push_neighbors"></a>
## push_neighbors

*`push_neighbors` add all successor states to the search stack. The list of successor state is gathered by [get_all_neighbors](#get_all_neighbors). Each successor state is hashed using [hash_state](#hash_state) and if it is not in the set of visited states, pushed onto the search stack.* **Uses** [get_all_neighbors](#get_all_neighbors) and [hash_state](#hash_state). **Used By** [forward_planner](#forward-planner)

* **state** List - list representation of S-expression(s) that make up the current state 
* **actions** Dict - dictionary containing possible actions, their preconditions, add_list and del_list
* **plan** List - list of actions taken by the search
* **path** List - list of states searched by the planner
* **stack** List - list of tuples, each tuple contains a state, plan and path
* **visited** set - set of visited states visited


**returns** List - modified search stack

In [ ]:
def push_neighbors(state: List, actions: Dict, plan: List, path: List, stack: List, visited: Set):
    for new_state, action in get_all_neighbors(state, actions):
        if hash_state(new_state) not in visited:
            stack.append((new_state, plan + [action], path + [state]))
    return stack

<a id="parse_inputs"></a>
## parse_inputs

*`parse_inputs` parse S-expression strings from a state, goal and actions dictionary into list representation of S-expressions. If parsing of an input fails, None is returned instead. The actions dictionary is also checked to ensure it contains the correct keys (action, conditions, add, delete).* **Used By** [forward_planner](#forward_planner)

* **state** List -  list of strings to be parsed as S-expressions
* **goal** List - list of strings to be parsed as S-expressions
* **actions** Dict - dictionary containing possible actions, their preconditions, add list and delete list, all of which represented as S-expression string to be parsed


**returns** Tuple[List | None, List | None, Dict| None] - parsed state, goal and actions dict

In [ ]:
def parse_inputs(start_state: List, goal: List, actions: Dict) -> Tuple[List| None, List | None, Dict | None]:
    if not start_state:
        return None, goal, actions

    if not goal:
        return start_state, None, actions

    state = [parse(p) for p in start_state]
    goal =  [parse(p) for p in goal]

    parsed_actions = {}
    for name, action in actions.items():
        expected_keys = ["action", "conditions", "add", "delete"]
        if not all(k in action for k in expected_keys):
            return start_state, goal, None
        
        parsed_actions[name] = {"action": parse(action["action"]), "conditions": [parse(c) for c in action["conditions"]],
                                "add": [parse(a) for a in action["add"]],"delete": [parse(d) for d in action["delete"]]}

    return state, goal, parsed_actions

<a id="forward_planner"></a>
## forward_planner

*`forward_planner` Using nested BFS, find a plan of actions that transform the starting state into the goal state. The outer search * 

* **start_state** List -  list of strings to be parsed as S-expressions
* **goal** List - list of strings to be parsed as S-expressions
* **actions** Dict - dictionary containing possible actions, their preconditions, add list and delete list, all of which represented as S-expression string to be parsed
* **debug** bool - if False return list of actions, if True return states as well 

**returns** List | None - List of actions, and if debug is enabled, list of actions and states or None if input parsing fails

In [ ]:
def forward_planner(start_state: List , goal: List, actions: Dict, debug=False) -> List | None:
    start_state, goal, actions = parse_inputs(start_state, goal, actions) # type: ignore
    if start_state is None or goal is None or actions is None:
        return None

    stack = [(start_state, [], [])]
    visited = set()  # graph search

    while stack:
        state, plan, path = stack.pop(0)

        if hash_state(state) in visited:
            continue
        visited.add(hash_state(state))

        if is_goal_state(state, goal):
            return get_plan(state, plan, path, debug)
        
        stack = push_neighbors(state, actions, plan, path, stack, visited)
    return None

In [ ]:
start_state = [
    "(plane 1973)",
    "(plane 2749)",
    "(plane 97)",
    "(plane 1211)",
    "(airport SFO)",
    "(airport JFK)",
    "(airport ORD)",
    "(at 1973 SFO)",
    "(at 2749 JFK)",
    "(at 97 ORD)",
    "(at 1211 SFO)",
    "(fueled 1973)",
    "(unfueled 2749)",
    "(unfueled 97)",
    "(fueled 1211)"
]

actions = {
    "fly": {
        "action": "(fly ?plane ?from ?to)",
        "conditions": [
            "(plane ?plane)",
            "(airport ?to)",
            "(airport ?from)",
            "(at ?plane ?from)",
            "(fueled ?plane)"
        ],
        "add": [
            "(at ?plane ?to)",
            "(unfueled ?plane)"
        ],
        "delete": [
            "(at ?plane ?from)",
            "(fueled ?plane)"
        ]
    }, 
    "fuel": {
        "action": "(fuel ?plane)",
        "conditions": [
            "(plane ?plane)",
            "(unfueled ?plane)",
        ],
        "add": [
            "(fueled ?plane)"
        ],
        "delete": [
            "(unfueled ?plane)"
        ]
    }
}

goal = [
    "(at 1973 ORD)",
    "(at 1211 JFK)",
    "(at 2749 SFO)"
]


plan = forward_planner(start_state, goal, actions, debug=True)
for action in plan:
    print(action)

You will be solving the problem from above. Here is the start state:

In [ ]:
start_state = [
    "(item Saw)",
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",
    "(agent Me)",
    "(at Me Home)",
    "(at Saw Store)",
    "(at Drill Store)"
]

The goal state:

In [ ]:
goal = [
    "(item Saw)",    
    "(item Drill)",
    "(place Home)",
    "(place Store)",
    "(place Bank)",    
    "(agent Me)",
    "(at Me Home)",
    "(at Drill Me)",
    "(at Saw Store)"    
]

and the actions/operators:

In [ ]:
actions = {
    "drive": {
        "action": "(drive ?agent ?from ?to)",
        "conditions": [
            "(agent ?agent)",
            "(place ?from)",
            "(place ?to)",
            "(at ?agent ?from)"
        ],
        "add": [
            "(at ?agent ?to)"
        ],
        "delete": [
            "(at ?agent ?from)"
        ]
    },
    "buy": {
        "action": "(buy ?purchaser ?seller ?item)",
        "conditions": [
            "(item ?item)",
            "(place ?seller)",
            "(agent ?purchaser)",
            "(at ?item ?seller)",
            "(at ?purchaser ?seller)"
        ],
        "add": [
            "(at ?item ?purchaser)"
        ],
        "delete": [
            "(at ?item ?seller)"
        ]
    }
}

**Note** The facts for each state are really an ordered set. When comparing two states, you may need to convert them to a Set first.

In [ ]:
plan = forward_planner( start_state, goal, actions, debug=True)

In [ ]:
assert plan is not None
for el in plan:
    print(el)

## Before You Submit...

1. Did you provide output exactly as requested?
2. Did you re-execute the entire notebook? ("Restart Kernel and Rull All Cells...")
3. If you did not complete the assignment or had difficulty please explain what gave you the most difficulty in the Markdown cell below.
4. Did you change the name of the file to `jhed_id.ipynb`?

Do not submit any other files.